# Day 04: SQL Window Functions

Practice notebook for Day 4 of the Week 1 plan. It uses **DuckDB** so you can execute SQL locally from this notebook without setting up a database server.

Choose the project `.venv` kernel in VS Code. Run setup and sample-data cells first, then work through each section. For each prompt, write your explanation in the markdown space and complete the SQL query cell.

**Topics:** `OVER`, `PARTITION BY`, ranking ties, `LAG`/`LEAD`, month-over-month change, running totals, and rolling windows.

**Source:** [PostgreSQL Tutorial §3.5: Window Functions](https://www.postgresql.org/docs/current/tutorial-window.html).

## 1. Setup and sample data

These cells create small in-memory tables. They do not create or modify files.

In [ ]:
import duckdb

con = duckdb.connect()
print(duckdb.__version__)

In [ ]:
con.sql("""
CREATE TABLE employees (
    employee_id INTEGER,
    employee_name VARCHAR,
    department VARCHAR,
    salary INTEGER
)
""")

con.sql("""
INSERT INTO employees VALUES
    (1, 'Asha', 'Engineering', 120000),
    (2, 'Ben', 'Engineering', 110000),
    (3, 'Chen', 'Engineering', 110000),
    (4, 'Diya', 'Sales', 95000),
    (5, 'Eli', 'Sales', 90000),
    (6, 'Farah', 'Sales', 90000)
""")

con.sql("SELECT * FROM employees ORDER BY department, salary DESC").df()

## 2. Window functions versus `GROUP BY`

A window function calculates across related rows while keeping each input row in the result. `GROUP BY` condenses rows into one result per group.

Run both queries. Explain why the row counts differ.

> Your explanation:
>


In [ ]:
con.sql("""
SELECT
    employee_name,
    department,
    salary,
    AVG(salary) OVER (PARTITION BY department) AS department_avg
FROM employees
ORDER BY department, salary DESC, employee_id
""").df()

In [ ]:
con.sql("""
SELECT department, AVG(salary) AS department_avg
FROM employees
GROUP BY department
ORDER BY department
""").df()

## 3. Ranking and ties

For Engineering, salaries are 120000, 110000, 110000. Predict the ranks before running this query.

- `ROW_NUMBER()`: assigns a unique sequential number to each row.
- `RANK()`: tied rows share a rank; following rank numbers have gaps.
- `DENSE_RANK()`: tied rows share a rank; following ranks have no gaps.

> Predicted ranks for the three Engineering employees:
>


In [ ]:
con.sql("""
SELECT
    department,
    employee_name,
    salary,
    ROW_NUMBER() OVER (
        PARTITION BY department ORDER BY salary DESC, employee_id
    ) AS row_num,
    RANK() OVER (
        PARTITION BY department ORDER BY salary DESC
    ) AS salary_rank,
    DENSE_RANK() OVER (
        PARTITION BY department ORDER BY salary DESC
    ) AS dense_salary_rank
FROM employees
ORDER BY department, salary DESC, employee_id
""").df()

Why does the `ROW_NUMBER` window include `employee_id`, while the `RANK` and `DENSE_RANK` windows do not?

> Your explanation:
>


## 4. `LAG`, `LEAD`, and month-over-month change

First make one row per month. Then use `LAG` to bring the previous month’s revenue onto the current row.

The sample intentionally skips March so you can notice that `LAG` means previous **row**, which may not be the previous calendar month.

> What should the February and April previous-row values be? What does the missing month change?
>


In [ ]:
con.sql("""
CREATE TABLE monthly_sales (
    month_start DATE,
    revenue INTEGER
)
""")

con.sql("""
INSERT INTO monthly_sales VALUES
    ('2026-01-01', 100),
    ('2026-02-01', 130),
    ('2026-04-01', 160),
    ('2026-05-01', 120)
""")

con.sql("""
WITH month_comparison AS (
    SELECT
        month_start,
        revenue,
        LAG(revenue) OVER (ORDER BY month_start) AS previous_row_revenue,
        LEAD(revenue) OVER (ORDER BY month_start) AS next_row_revenue
    FROM monthly_sales
)
SELECT
    month_start,
    revenue,
    previous_row_revenue,
    next_row_revenue,
    revenue - previous_row_revenue AS change_from_previous_row,
    100.0 * (revenue - previous_row_revenue)
        / NULLIF(previous_row_revenue, 0) AS pct_change_from_previous_row
FROM month_comparison
ORDER BY month_start
""").df()

To calculate true calendar month-over-month values when months can be absent, what would you need to do before applying `LAG`?

> Your answer:
>


## 5. Running total and rolling window

These are related but distinct patterns:

- **Running total:** accumulates from the start through the current row.
- **Rolling window:** calculates over a fixed number of nearby rows, such as the current month and previous two months.

The explicit frame `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` contains up to three rows.

In [ ]:
con.sql("""
SELECT
    month_start,
    revenue,
    SUM(revenue) OVER (
        ORDER BY month_start
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total,
    AVG(revenue) OVER (
        ORDER BY month_start
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS rolling_3_row_avg
FROM monthly_sales
ORDER BY month_start
""").df()

The rolling average above is over three rows, not necessarily three calendar months if there are gaps. What could you change in the input data/query to make it a true rolling three-month average?

> Your answer:
>


## 6. LeetCode #178 — Rank Scores

Complete the SQL query. The required ranking is dense: tied scores share a rank, and the next distinct score gets the next integer rank.

The table is intentionally small and includes a tie.

In [ ]:
con.sql("""
CREATE TABLE scores (score DECIMAL(4,2))
""")
con.sql("""
INSERT INTO scores VALUES
    (3.50), (3.65), (4.00), (3.85), (4.00), (3.65)
""")

In [ ]:
query_178 = """
-- Replace this comment with your SQL query.
"""

con.sql(query_178).df()

## 7. LeetCode #185 — Department Top Three Salaries

Return employees whose **distinct salary rank** within their department is at most 3. This means ties at the third salary rank are included.

Write a query using `DENSE_RANK() OVER (PARTITION BY department ...)`. You can use the `employees` table from section 1.

In [ ]:
query_185 = """
-- Replace this comment with your SQL query.
"""

con.sql(query_185).df()

## 8. Review

Explain these in your own words:

1. Why can a window query show each employee and the department average on the same row?
2. On tied salaries, how do `ROW_NUMBER`, `RANK`, and `DENSE_RANK` differ?
3. What does `LAG` return when calendar months are missing?
4. How does a rolling three-row average differ from a running total?
5. Which function and partitioning would you use for LeetCode #185?

> Notes:
>


## 9. Answer key

Reference solutions for review after attempting the cells above. These use the sample tables created in this notebook.

### #178 — Rank Scores

`DENSE_RANK` assigns equal scores the same rank and leaves no gaps after ties.

In [ ]:
con.sql("""
SELECT
    score,
    DENSE_RANK() OVER (ORDER BY score DESC) AS "rank"
FROM scores
ORDER BY score DESC
""").df()

### #185 — Department Top Three Salaries

First rank distinct salaries inside each department; then keep ranks 1 through 3. Filtering the ranked rows in an outer query is necessary because window functions are evaluated after `WHERE`.

In [ ]:
con.sql("""
WITH ranked_employees AS (
    SELECT
        department,
        employee_name,
        salary,
        DENSE_RANK() OVER (
            PARTITION BY department
            ORDER BY salary DESC
        ) AS salary_rank
    FROM employees
)
SELECT department, employee_name, salary
FROM ranked_employees
WHERE salary_rank <= 3
ORDER BY department, salary DESC, employee_name
""").df()

### Reference patterns

- Department average while retaining employees: `AVG(salary) OVER (PARTITION BY department)`.
- Previous/next row: `LAG(value) OVER (PARTITION BY group ORDER BY period)` / `LEAD(...)`.
- Month-over-month change: aggregate to one row per month first, then subtract `LAG(monthly_value)` from the current monthly value. For missing months, join against a complete calendar month series before using `LAG`.
- Running sum: `SUM(value) OVER (ORDER BY period ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`.
- Three-row rolling average: `AVG(value) OVER (ORDER BY period ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)`. For calendar-month windows, ensure the input has one row per month.